In [ ]:
from google.colab import drive
drive.mount('/content/drive')
exec(open("/content/drive/MyDrive/imdb_peft_project/code/lora-imdb-classifier/00_colab_setup.py").read())

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ /content/drive/MyDrive/imdb_peft_project
✓ /content/drive/MyDrive/imdb_peft_project/checkpoints/roberta_lora
✓ /content/drive/MyDrive/imdb_peft_project/checkpoints/deberta_lora
✓ /content/drive/MyDrive/imdb_peft_project/oof_predictions
✓ /content/drive/MyDrive/imdb_peft_project/results
✓ /content/drive/MyDrive/imdb_peft_project/notebooks
✓ /content/drive/MyDrive/imdb_peft_project/code

Folder structure ready.
Enter GitHub Token: ··········
Repository exists, pulling latest changes...
✓ Pull complete.

Repository path: /content/drive/MyDrive/imdb_peft_project/code/lora-imdb-classifier
SETUP COMPLETE
Drive folder  : /content/drive/MyDrive/imdb_peft_project
GitHub repo   : /content/drive/MyDrive/imdb_peft_project/code/lora-imdb-classifier

Available functions:
  push_to_github('message')        → push code to GitHub
  save_code_to_rep

In [ ]:
!pip install transformers peft accelerate torchao --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 13.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import time
import os
from transformers import (DebertaV2Tokenizer, DebertaV2ForSequenceClassification,
                          TrainingArguments, Trainer, TrainerCallback)
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score, roc_auc_score)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [ ]:
DIRS["checkpoints2_best"] = f"{DIRS['root']}/checkpoints/deberta_lora_v2_best"
os.makedirs(DIRS["checkpoints2_best"], exist_ok=True)
print(f"✓ {DIRS['checkpoints2_best']}")

train_df = pd.read_parquet(f"{DIRS['root']}/train_df_v2.parquet")
test_df  = pd.read_parquet(f"{DIRS['root']}/test_df_v2.parquet")
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

✓ /content/drive/MyDrive/imdb_peft_project/checkpoints/deberta_lora_v2_best
Train: 25000 | Test: 25000


In [ ]:
def head_tail_truncate_v2(text, tokenizer, max_len=512, head_len=256):
    """V2: Equal split — first 256 + last 256 tokens."""
    tail_len = max_len - head_len
    tokens = tokenizer(text, add_special_tokens=False,
                       truncation=False, return_tensors=None)
    input_ids      = tokens["input_ids"]
    attention_mask = tokens["attention_mask"]
    if len(input_ids) > max_len - 2:
        input_ids      = input_ids[:head_len] + input_ids[-tail_len:]
        attention_mask = attention_mask[:head_len] + attention_mask[-tail_len:]
    return tokenizer(
        tokenizer.decode(input_ids),
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors=None
    )

class IMDBDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512, head_len=256):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.head_len  = head_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text  = self.df.loc[idx, "text_clean"]
        label = self.df.loc[idx, "label"]
        encoding = head_tail_truncate_v2(
            text, self.tokenizer, self.max_len, self.head_len
        )
        return {
            "input_ids":      torch.tensor(encoding["input_ids"],      dtype=torch.long),
            "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
            "labels":         torch.tensor(label,                      dtype=torch.long),
        }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy" : accuracy_score(labels, preds),
        "f1"       : f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall"   : recall_score(labels, preds),
        "roc_auc"  : roc_auc_score(labels, probs)
    }

print("Loading DeBERTa tokenizer...")
tokenizer     = DebertaV2Tokenizer.from_pretrained("microsoft/deberta-v3-base")
train_dataset = IMDBDataset(train_df, tokenizer)
test_dataset  = IMDBDataset(test_df,  tokenizer)
print(f"✓ Train: {len(train_dataset)} — Test: {len(test_dataset)}")

Loading DeBERTa tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

✓ Train: 25000 — Test: 25000


In [ ]:
class SaveCallbackFixed(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = int(state.epoch)
        save_path = f"{DIRS['checkpoints2_best']}/epoch_{epoch}"
        os.makedirs(save_path, exist_ok=True)
        kwargs["model"].save_pretrained(save_path)
        extra_state = {
            "classifier.weight": kwargs["model"].base_model.model.classifier.weight.data.cpu(),
            "classifier.bias"  : kwargs["model"].base_model.model.classifier.bias.data.cpu(),
            "pooler.weight"    : kwargs["model"].base_model.model.pooler.dense.weight.data.cpu(),
            "pooler.bias"      : kwargs["model"].base_model.model.pooler.dense.bias.data.cpu(),
        }
        torch.save(extra_state, f"{save_path}/extra_weights.pt")
        print(f"✓ Epoch {epoch} saved — LoRA + classifier + pooler.")

print("Loading DeBERTa...")
model = DebertaV2ForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=2,
    torch_dtype=torch.float32
)

# Best config: r=32, lr=5e-5
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=["query_proj", "value_proj"],
    bias="none"
)

model = get_peft_model(model, lora_config)
model = model.to(torch.float32).to(device)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=f"{DIRS['checkpoints2_best']}",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=False,
    bf16=False,
    seed=SEED,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[SaveCallbackFixed()]
)

print("Training started...")
start = time.time()
trainer.train()
train_time = time.time() - start
print(f"\n✓ Training complete. Duration: {train_time/60:.1f} minutes")

Loading DeBERTa...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

...

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Roc Auc
1,0.322276,0.127922,0.954720,0.955186,0.945455,0.965120,0.989495


✓ Epoch 1 saved — LoRA + classifier + pooler.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Roc Auc
1,0.322276,0.127922,0.954720,0.955186,0.945455,0.965120,0.989495
2,0.297777,0.134714,0.958600,0.958452,0.961889,0.955040,0.991122
3,0.281755,0.131854,0.959240,0.959549,0.952328,0.966880,0.991352


✓ Epoch 2 saved — LoRA + classifier + pooler.
✓ Epoch 3 saved — LoRA + classifier + pooler.

✓ Training complete. Duration: 50.4 minutes


In [ ]:
save_results({
    "model"      : "DeBERTa-v3 + LoRA Best (r=32, lr=5e-5)",
    "accuracy"   : 0.9592,
    "f1"         : 0.9595,
    "precision"  : 0.9523,
    "recall"     : 0.9669,
    "roc_auc"    : 0.9914,
    "epochs"     : 3,
    "r"          : 32,
    "lr"         : 5e-5,
    "lora_alpha" : 64,
    "head_tail"  : "256/256",
    "fp16"       : False,
}, "deberta_lora_v2_best.json")

model.save_pretrained(f"{DIRS['checkpoints2_best']}/final")
tokenizer.save_pretrained(f"{DIRS['checkpoints2_best']}/final")
torch.save({
    "classifier.weight": model.base_model.model.classifier.weight.data.cpu(),
    "classifier.bias"  : model.base_model.model.classifier.bias.data.cpu(),
    "pooler.weight"    : model.base_model.model.pooler.dense.weight.data.cpu(),
    "pooler.bias"      : model.base_model.model.pooler.dense.bias.data.cpu(),
}, f"{DIRS['checkpoints2_best']}/final/extra_weights.pt")
print("✓ Model saved to Drive.")

NOTEBOOK_NAME = "04d_deberta_lora_v2_best"
!jupyter nbconvert --to script \
  "/content/drive/MyDrive/Colab Notebooks/{NOTEBOOK_NAME}.ipynb" \
  --output-dir "/content/"
import os
os.rename(f"/content/{NOTEBOOK_NAME}.txt",
          f"/content/{NOTEBOOK_NAME}.py")
save_code_to_repo(f"/content/{NOTEBOOK_NAME}.py")
push_to_github("deberta v2 best r32 lr5e5 acc 0.9592")

✓ Results saved: /content/drive/MyDrive/imdb_peft_project/results/deberta_lora_v2_best.json
✓ Model saved to Drive.
[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/04d_deberta_lora_v2_best.ipynb to script
[NbConvertApp] ERROR | Notebook JSON is invalid: Additional properties are not allowed ('metadata' was unexpected)

Failed validating 'additionalProperties' in stream:

On instance['cells'][5]['outputs'][0]:
{'metadata': {'tags': None},
 'name': 'stdout',
 'output_type': 'stream',
 'text': 'Loading DeBERTa...\n'}
[NbConvertApp] Writing 5610 bytes to /content/04d_deberta_lora_v2_best.txt
✓ 04d_deberta_lora_v2_best.py → copied to repository.
✓ Pushed to GitHub: 'deberta v2 best r32 lr5e5 acc 0.9592'
